## 1. Import Required Libraries

The required libraries are imported to support file handling, numerical operations,
embedding generation, vector similarity search, and environment configuration.

- **Embedding:** Numerical representation of text that captures semantic meaning.
- **FAISS:** Library used for efficient similarity search over vectors.
- **Sentence Transformers:** Models that convert text into embeddings.

In [2]:
import os
import numpy as np
import faiss

# Path helps us work with folders and files in a platform-independent way
from pathlib import Path

# SentenceTransformer converts text into numerical embeddings
from sentence_transformers import SentenceTransformer

# Used later to load environment variables such as API keys
from dotenv import load_dotenv

# SSL certificate configuration:
# Uses the Linux system's trusted CA certificates for HTTPS connections.
# This is required in our corporate network because the default Python
# certificate bundle does not trust the network's certificate chain.
os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

## 2. Define Data Paths

The paths to client documents and previous meeting notes are defined so the
retrieval pipeline can access and organize the different information sources.

- **Pathlib:** Python library for working with file and directory paths.

In [3]:
# Define the main project directory.
# '..' means move one level up from the notebooks folder.
BASE_DIR = Path('..')

# Location of documents containing general client information
CLIENT_DOCS_DIR = BASE_DIR / 'data' / 'client_documents'

# Location of documents containing previous meeting notes
MEETING_NOTES_DIR = BASE_DIR / 'data' / 'meeting_notes'

## 3. Load Source Documents

Client information and previous meeting notes are loaded into memory as structured
records so they can be processed by the retrieval system.

- **Metadata:** Information such as source name and document type stored alongside content.

In [4]:
documents = []

# Read all client documents from the client_documents folder
for file_path in CLIENT_DOCS_DIR.glob('*.txt'):

    documents.append({
        'source': file_path.name,              # Store the original filename
        'type': 'client_document',             # Identify the document category
        'text': file_path.read_text()          # Read the document contents
    })

# Read all previous meeting notes
for file_path in MEETING_NOTES_DIR.glob('*.txt'):

    documents.append({
        'source': file_path.name,              # Store the original filename
        'type': 'meeting_note',                # Identify the document category
        'text': file_path.read_text()          # Read the document contents
    })

# Check how many documents were successfully loaded
len(documents)

6

## 4. Document Chunking

Documents are divided into smaller sections to improve retrieval precision and
allow the system to retrieve specific relevant information.

- **Chunk:** Smaller section of a document used for retrieval.
- **Overlap:** Shared content between consecutive chunks that helps preserve context.

In [5]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []

    # Start from the beginning of the document
    start = 0

    # Continue creating chunks until we reach the end of the text
    while start < len(text):

        # Define where the current chunk should end
        end = start + chunk_size

        # Store the current piece of text
        chunks.append(text[start:end])

        # Move forward while keeping the required overlap
        start += chunk_size - overlap

    return chunks

## 5. Create the Chunk Collection

Each document is split into chunks while preserving its source and document type,
creating the searchable collection used by the embedding pipeline.

In [6]:
chunks = []

# Process every document that we loaded earlier
for document in documents:

    # Split the document text into smaller chunks
    document_chunks = create_chunks(document['text'])

    # Store each chunk together with its metadata
    for chunk in document_chunks:

        chunks.append({
            'source': document['source'],      # Original document name
            'type': document['type'],           # Document category
            'text': chunk                       # Chunk content
        })

# Check how many chunks were created
len(chunks)

12

## 6. Initialize the Embedding Model

The embedding model converts textual chunks into numerical vectors that can be
compared based on semantic similarity.

- **Embedding:** Numerical vector representation of text.
- **Vector space:** Mathematical space in which embeddings are compared.

In [7]:
# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4407.17it/s]


## 7. Generate Document Embeddings

Each chunk is converted into a 384-dimensional vector so that its semantic
relationship with a user query can be measured during retrieval.

In [8]:
# Extract only the text from each chunk
texts = [chunk['text'] for chunk in chunks]

# Convert every text chunk into a numerical embedding
embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

# Display the shape of our embedding matrix
embeddings.shape

(12, 384)

## 8. Create the FAISS Vector Index

The generated embeddings are stored in a FAISS index, enabling efficient similarity
search when relevant information needs to be retrieved.

- **Vector index:** Structure used to organize vectors for similarity search.
- **L2 distance:** Euclidean distance used to measure the difference between vectors.

In [9]:
# Get the number of dimensions from our embedding matrix
dimension = embeddings.shape[1]

# Create a FAISS index using Euclidean (L2) distance
index = faiss.IndexFlatL2(dimension)

# Add all document chunk embeddings to the vector index
index.add(embeddings)

# Check how many vectors are currently stored
print('Number of vectors:', index.ntotal)

Number of vectors: 12


## 9. Implement Semantic Document Retrieval

The retrieval function converts a user query into an embedding and searches the
FAISS index for the most similar document chunks.

- **Semantic search:** Retrieval based on meaning rather than exact keyword matching.
- **Top-k:** Number of most relevant results returned by the search.

In [10]:
def search_documents(query, k=3):

    # Convert the user's query into the same embedding space
    # used for our document chunks
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search FAISS for the k closest vectors
    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    # Match each retrieved vector with its original chunk
    for distance, idx in zip(distances[0], indices[0]):

        results.append({
            'source': chunks[idx]['source'],       # Original document
            'type': chunks[idx]['type'],           # Document category
            'text': chunks[idx]['text'],           # Retrieved content
            'distance': float(distance)            # Similarity distance
        })

    return results

## 10. Validate Semantic Retrieval

Retrieval is tested independently to verify that relevant client information and
meeting notes can be identified before connecting the search mechanism to the agent.

In [11]:
# Test our semantic search with a realistic user question
results = search_documents(
    'What are the current issues with Acme Corp?'
)

# Display each retrieved result
for result in results:

    print('SOURCE:', result['source'])
    print('DISTANCE:', result['distance'])
    print(result['text'])
    print('-' * 50)

SOURCE: company_overview.txt
DISTANCE: 0.7414597272872925
Acme Corp is an e-commerce technology company focused on helping online retailers
manage orders, customers, and operational workflows.

The company is currently investing in improving its analytics capabilities.
Leadership wants better visibility into customer behavior, operational performance,
and business KPIs.

Acme Corp is evaluating improvements to its existing analytics platform and is
particularly interested in faster reporting, better dashboards, and improved
integration with internal sy
--------------------------------------------------
SOURCE: meeting_2026_06_10.txt
DISTANCE: 0.8798907995223999
Meeting Date: 2026-06-10

Client: Acme Corp

Attendees:
- Sarah Mitchell — VP of Operations, Acme Corp
- Alex Johnson — Account Manager

Discussion:
The teams discussed Acme Corp's need to modernize its analytics platform.
Acme identified slow reporting and limited KPI visibility as major issues.

Sarah requested a proposal cove

## 11. Create Separate Search Functions

The agent needs different tools for different types of information. Separate search
functions allow it to retrieve either client information or previous meeting notes.

- **Tool:** A function the agent can call to perform a specific task.

In [12]:
def search_client_documents(query, k=3):
    # Get only client-document chunks
    client_chunks = [
        chunk for chunk in chunks
        if chunk['type'] == 'client_document'
    ]

    # Create embeddings for the client-document chunks
    client_embeddings = model.encode(
        [chunk['text'] for chunk in client_chunks],
        convert_to_numpy=True
    )

    # Create a FAISS index for client documents
    client_index = faiss.IndexFlatL2(client_embeddings.shape[1])
    client_index.add(client_embeddings)

    # Convert the query into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search for the most similar chunks
    distances, indices = client_index.search(
        query_embedding,
        k
    )

    results = []

    # Return the matching chunks with their source
    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            'source': client_chunks[idx]['source'],
            'text': client_chunks[idx]['text'],
            'distance': float(distance)
        })

    return results

## 12. Search Previous Meeting Notes

Previous meeting notes contain discussions, decisions, and action items that are
important for preparing for a client meeting.

In [13]:
def search_meeting_notes(query, k=3):
    # Get only meeting-note chunks
    meeting_chunks = [
        chunk for chunk in chunks
        if chunk['type'] == 'meeting_note'
    ]

    # Create embeddings for the meeting-note chunks
    meeting_embeddings = model.encode(
        [chunk['text'] for chunk in meeting_chunks],
        convert_to_numpy=True
    )

    # Create a FAISS index for meeting notes
    meeting_index = faiss.IndexFlatL2(meeting_embeddings.shape[1])
    meeting_index.add(meeting_embeddings)

    # Convert the query into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search for the most similar meeting-note chunks
    distances, indices = meeting_index.search(
        query_embedding,
        k
    )

    results = []

    # Return the matching chunks with their source
    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            'source': meeting_chunks[idx]['source'],
            'text': meeting_chunks[idx]['text'],
            'distance': float(distance)
        })

    return results

In [14]:
# Test client information retrieval
client_results = search_client_documents(
    'What are Acme Corp current concerns?'
)

for result in client_results:
    print(result['source'])
    print(result['text'])
    print('-' * 50)

company_overview.txt
Acme Corp is an e-commerce technology company focused on helping online retailers
manage orders, customers, and operational workflows.

The company is currently investing in improving its analytics capabilities.
Leadership wants better visibility into customer behavior, operational performance,
and business KPIs.

Acme Corp is evaluating improvements to its existing analytics platform and is
particularly interested in faster reporting, better dashboards, and improved
integration with internal sy
--------------------------------------------------
client_profile.txt
Client: Acme Corp

Industry: E-commerce Technology

Headquarters: Austin, Texas

Company Size: Approximately 500 employees

Primary Contact: Sarah Mitchell

Role: VP of Operations

Relationship: Existing client

Account Status: Active

Key Business Areas:
- E-commerce operations
- Customer analytics
- Order management
- Business intelligence

Client Preferences:
- Prefers concise presentations
- Values da

In [15]:
# Test previous meeting retrieval
meeting_results = search_meeting_notes(
    'What action items are pending?'
)

for result in meeting_results:
    print(result['source'])
    print(result['text'])
    print('-' * 50)

meeting_2026_08_12.txt
dentified
additional integration work that may affect the timeline.

Pricing for the next phase was also discussed. Acme requested a revised pricing
proposal.

Action Items:
- Alex: Send revised pricing proposal.
- David: Confirm API integration timeline.
- Sarah: Provide final API documentation.

Status:
Dashboard: Prototype approved.
API Integration: In progress.
Pricing: Pending.
--------------------------------------------------
meeting_2026_07_15.txt
Sarah requested a working dashboard prototype.

Action Items:
- Alex: Provide dashboard prototype.
- David: Evaluate API integration requirements.
- Sarah: Review proposed reporting KPIs.

Status:
Proposal reviewed.
Dashboard prototype in development.
API requirements being evaluated.
--------------------------------------------------
meeting_2026_06_10.txt
Meeting Date: 2026-06-10

Client: Acme Corp

Attendees:
- Sarah Mitchell — VP of Operations, Acme Corp
- Alex Johnson — Account Manager

Discussion:
The team

## 13. Short-Term Memory

Short-term memory stores the current conversation so the agent can use previous
messages while completing the same task.

- **Short-term memory:** Temporary conversation context maintained during a session.

In [16]:
# Store the current conversation messages
conversation_history = []

## 14. Add Conversation Messages

Each user message and agent response is added to the conversation history so that
the agent can maintain context throughout the current session.

In [17]:
def add_to_memory(role, content):
    # Store each message with its role and content
    conversation_history.append({
        'role': role,
        'content': content
    })

In [18]:
# Add a sample conversation
add_to_memory('user', 'Prepare me for my meeting with Acme Corp.')

add_to_memory(
    'assistant',
    'I will gather the client information and previous meeting notes.'
)

# Display the current conversation
conversation_history

[{'role': 'user', 'content': 'Prepare me for my meeting with Acme Corp.'},
 {'role': 'assistant',
  'content': 'I will gather the client information and previous meeting notes.'}]

## 15. Long-Term Memory

Long-term memory stores information that should remain available across different
sessions. A vector database allows stored information to be retrieved later based
on semantic similarity.

- **Long-term memory:** Persistent information retained beyond the current session.
- **Persistent storage:** Data that remains available after the session ends.

In [19]:
# Store long-term memory entries separately from the document chunks
long_term_memories = []

# Create a separate FAISS index for long-term memory
memory_index = faiss.IndexFlatL2(384)

## 16. Store Long-Term Memories

Important information provided during a conversation can be stored as memory.
Each memory is converted into an embedding and added to the persistent memory index.

In [20]:
def store_memory(memory):
    # Save the original memory text
    long_term_memories.append(memory)

    # Convert the memory into an embedding
    memory_embedding = model.encode(
        [memory],
        convert_to_numpy=True
    )

    # Add the embedding to the memory index
    memory_index.add(memory_embedding)

In [21]:
# Store information that may be useful in future sessions
store_memory('Sarah Mitchell prefers concise presentations.')

store_memory('Acme Corp values data-driven recommendations.')

print('Stored memories:', len(long_term_memories))

Stored memories: 2


## 17. Retrieve Long-Term Memories

Stored memories are retrieved using semantic similarity, allowing the agent to find
relevant information even when the query does not exactly match the stored text.

In [22]:
def search_memory(query, k=3):
    # Convert the query into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search the long-term memory index
    distances, indices = memory_index.search(
        query_embedding,
        min(k, len(long_term_memories))
    )

    results = []

    # Return the matching memories
    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            'memory': long_term_memories[idx],
            'distance': float(distance)
        })

    return results

In [23]:
# Test long-term memory retrieval
memory_results = search_memory(
    'What does Sarah prefer during presentations?'
)

for result in memory_results:
    print(result['memory'])

Sarah Mitchell prefers concise presentations.
Acme Corp values data-driven recommendations.


## 18. Define Agent Tools

The retrieval functions are exposed as tools that the agent can call when additional
information is required to complete the user's request.

- **Tool:** A function that allows an agent to perform a specific action.

In [24]:
# Store the available tools in a dictionary
# The agent will use these names to decide which function to call
tools = {
    'search_client_documents': search_client_documents,
    'search_meeting_notes': search_meeting_notes,
    'search_memory': search_memory
}

# Display the available tools
list(tools.keys())

['search_client_documents', 'search_meeting_notes', 'search_memory']

## 19. Configure the LLM

The LLM acts as the reasoning component of the agent. It interprets the user's goal,
decides which tools are required, and uses the retrieved information to produce the
final response.

- **LLM:** Language model that processes instructions and generates responses.
- **Agent:** System that uses an LLM together with tools to achieve a specific goal.

In [25]:
# Load environment variables from the .env file
load_dotenv()

# Retrieve the Gemini API key securely from the environment
api_key = os.getenv('GEMINI_API_KEY')

# Check that the API key is available
if not api_key:
    raise ValueError('GEMINI_API_KEY not found in .env')

In [26]:
import google.generativeai as genai

# Configure the Gemini client with the API key
genai.configure(api_key=api_key)

# Initialize the model used by the agent
llm = genai.GenerativeModel('gemini-3.5-flash-lite')

/tmp/ipykernel_617359/2417654049.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 20. Define the Agent Instructions

The system prompt defines the agent's objective, available tools, and expected
behavior. The instructions ensure that the agent retrieves information before
preparing the meeting brief instead of responding as a simple Q&A chatbot.

- **System prompt:** Instructions that define the behavior and role of the agent.
- **Agentic workflow:** A process in which the agent selects and uses tools to complete a goal.

In [27]:
# Define the agent's role and expected behavior
agent_prompt = '''
You are a client meeting preparation agent.

Your goal is to prepare concise meeting briefs using the available information.

Available tools:
1. search_client_documents - retrieves client and project information.
2. search_meeting_notes - retrieves previous meeting discussions and action items.
3. search_memory - retrieves relevant long-term memories.

For a meeting preparation request:
- Retrieve relevant client information.
- Retrieve previous meeting information.
- Retrieve relevant long-term memories when useful.
- Identify important topics, open action items, concerns, and next steps.
- Use retrieved information to create a concise and factual meeting brief.
- Do not invent information that is not present in the retrieved data.
'''

## 21. Define the Agent's Tool-Calling Logic

The agent needs a structured way to select a tool, provide its input, and receive the
result. This creates the connection between the LLM's decision-making and the
retrieval functions.

- **Tool call:** A request from the agent to execute a specific function.
- **Observation:** The result returned by a tool after execution.

In [28]:
def execute_tool(tool_name, query):
    # Check whether the requested tool exists
    if tool_name not in tools:
        return 'Tool not available.'

    # Execute the selected tool with the user's query
    result = tools[tool_name](query)

    return result

In [29]:
# Test the client-document tool
tool_result = execute_tool(
    'search_client_documents',
    'What are Acme Corp current concerns?'
)

# Display the retrieved information
for result in tool_result:
    print(result['text'])
    print('-' * 50)

Acme Corp is an e-commerce technology company focused on helping online retailers
manage orders, customers, and operational workflows.

The company is currently investing in improving its analytics capabilities.
Leadership wants better visibility into customer behavior, operational performance,
and business KPIs.

Acme Corp is evaluating improvements to its existing analytics platform and is
particularly interested in faster reporting, better dashboards, and improved
integration with internal sy
--------------------------------------------------
Client: Acme Corp

Industry: E-commerce Technology

Headquarters: Austin, Texas

Company Size: Approximately 500 employees

Primary Contact: Sarah Mitchell

Role: VP of Operations

Relationship: Existing client

Account Status: Active

Key Business Areas:
- E-commerce operations
- Customer analytics
- Order management
- Business intelligence

Client Preferences:
- Prefers concise presentations
- Values data-driven recommendations
- Interested i

## 22. Implement the ReAct Agent Loop

The agent follows a **Reason → Act → Observe** cycle. It determines what information
is required, calls the appropriate tool, and uses the returned result as context for
the next decision.

- **ReAct:** Reasoning and action combined in an iterative workflow.
- **Iteration:** One cycle of reasoning, tool execution, and observation.

## 24. Return the Agent History

The agent history contains the user's request, tool calls, and tool results generated
during execution. Returning this history makes the agentic workflow observable and
allows the reasoning process to be inspected without exposing private model reasoning.

- **Agent history:** Sequence of messages and tool observations generated during execution.

In [30]:
MAX_ITERATIONS = 5

def run_agent(user_query):
    # Initialize short-term memory with the user's request
    history = [
        {
            'role': 'user',
            'content': user_query
        }
    ]

    for iteration in range(MAX_ITERATIONS):

        # Combine the conversation history into the context sent to the LLM
        context = '\n'.join(
            f"{message['role']}: {message['content']}"
            for message in history
        )

        # Ask the LLM to decide whether another tool is required
        response = llm.generate_content(
            f'''
            {agent_prompt}

            Conversation:
            {context}

            Decide what information is needed next.

            If a tool is required, respond using exactly:

            TOOL: tool_name
            QUERY: search query

            If enough information has been gathered, respond using exactly:

            FINAL: answer
            '''
        )

        response_text = response.text.strip()

        # Stop when the agent produces the final answer
        if response_text.startswith('FINAL:'):
            final_answer = response_text.replace(
                'FINAL:', '', 1
            ).strip()

            return final_answer, history

        # Process a tool request
        if response_text.startswith('TOOL:'):

            lines = response_text.splitlines()

            # Extract the selected tool and search query
            tool_name = lines[0].replace('TOOL:', '').strip()
            query = lines[1].replace('QUERY:', '').strip()

            # Execute the selected tool
            observation = execute_tool(tool_name, query)

            # Store the tool call in short-term memory
            history.append({
                'role': 'assistant',
                'content': response_text
            })

            # Store the tool result as an observation
            history.append({
                'role': 'tool',
                'content': str(observation)
            })

    # Return the history even when the maximum iteration limit is reached
    return (
        'Unable to complete the task within the maximum number of iterations.',
        history
    )

## 25. Inspect the Agent Workflow

The agent is executed again and its history is displayed alongside the final response.
This provides a visible record of the tool calls and observations used to complete
the task.

In [31]:
# Run the meeting preparation agent
meeting_brief, agent_history = run_agent(
    'Prepare me for my meeting with Acme Corp.'
)

# Display the final meeting brief
print('MEETING BRIEF')
print('=' * 50)
print(meeting_brief)

print('\nAGENT WORKFLOW')
print('=' * 50)

# Display the messages and tool observations generated during execution
for message in agent_history:
    print(f"{message['role'].upper()}:")
    print(message['content'])
    print('-' * 50)

MEETING BRIEF
# Meeting Brief: Acme Corp

## Client Overview
- **Company:** Acme Corp (E-commerce Technology, ~500 employees)
- **Primary Contact:** Sarah Mitchell, VP of Operations
- **Relationship:** Existing client, active analytics modernization initiative
- **Preferences:** Concise presentations, data-driven recommendations, measurable business outcomes

## Previous Meeting & Project Status (August 12, 2026)
- **Dashboard:** Prototype demonstrated and approved by Sarah.
- **API Integration:** In progress; technical team identified additional integration work that may affect the timeline.
- **Pricing:** Pending revised pricing proposal.

## Open Action Items
- **Alex:** Send revised pricing proposal.
- **David:** Confirm API integration timeline.
- **Sarah:** Provide final API documentation.

## Key Topics & Next Steps for Upcoming Meeting
- Review and finalize the revised pricing proposal for the next phase.
- Discuss the updated API integration timeline based on the technical rev

## 26. Make Long-Term Memory Persistent

Long-term memory should remain available after the current session ends. Saving
memories to a file provides persistent storage that can be loaded again in a future
session.

- **Persistence:** Data that remains available after a session ends.

In [32]:
# Define the file used to persist long-term memories
MEMORY_FILE = BASE_DIR / 'data' / 'memory' / 'long_term_memory.txt'

## 27. Save Long-Term Memories

Each stored memory is written to persistent storage so that it can be recovered
during a later session.

In [33]:
def save_memory(memory):
    # Avoid storing duplicate memories
    if memory in long_term_memories:
        return

    # Add the memory to the current session
    long_term_memories.append(memory)

    # Create an embedding for the new memory
    memory_embedding = model.encode(
        [memory],
        convert_to_numpy=True
    )

    # Add the embedding to FAISS
    memory_index.add(memory_embedding)

    # Save the memory for future sessions
    with open(MEMORY_FILE, 'a') as file:
        file.write(memory + '\n')

## 28. Load Long-Term Memories

Previously stored memories are loaded when the application starts. Their embeddings
are regenerated and added to the memory index so they can be retrieved during a new
session.

In [34]:
def load_memories():
    # Check whether the memory file exists
    if not MEMORY_FILE.exists():
        return

    # Read previously stored memories
    with open(MEMORY_FILE, 'r') as file:
        memories = [
            line.strip()
            for line in file
            if line.strip()
        ]

    # Add each stored memory to the current memory system
    for memory in memories:
        long_term_memories.append(memory)

    # Generate embeddings for all stored memories
    if long_term_memories:
        memory_embeddings = model.encode(
            long_term_memories,
            convert_to_numpy=True
        )

        # Add the embeddings to the FAISS memory index
        memory_index.add(memory_embeddings)

In [35]:
# Load memories saved from previous sessions
load_memories()

print('Loaded memories:', len(long_term_memories))

Loaded memories: 8


## 29. Integrate Long-Term Memory into the Agent

The agent should be able to retrieve relevant information from long-term memory when
preparing a meeting brief. This allows information from previous sessions to contribute
to the current task.

In [36]:
agent_prompt = '''
You are a client meeting preparation agent.

Your goal is to prepare concise meeting briefs using the available information.

Available tools:
1. search_client_documents - retrieves client and project information.
2. search_meeting_notes - retrieves previous meeting discussions and action items.
3. search_memory - retrieves relevant long-term memories.

For a meeting preparation request:
- Retrieve relevant client information.
- Retrieve previous meeting information.
- Retrieve relevant long-term memories when useful.
- Identify important topics, open action items, concerns, and next steps.
- Use retrieved information to create a concise and factual meeting brief.
- Do not invent information that is not present in the retrieved data.
'''

In [37]:
tools = {
    'search_client_documents': search_client_documents,
    'search_meeting_notes': search_meeting_notes,
    'search_memory': search_memory
}

## 30. Test Long-Term Memory

The agent is tested with a meeting-preparation request to verify that long-term memory
is available as one of its information sources.

In [38]:
meeting_brief, agent_history = run_agent(
    'Prepare me for my meeting with Acme Corp.'
)

print(meeting_brief)

# Meeting Brief: Acme Corp

## Client Overview
- **Company:** Acme Corp (E-commerce Technology, ~500 employees)
- **Headquarters:** Austin, Texas
- **Primary Contact:** Sarah Mitchell, VP of Operations
- **Account Status:** Active, existing client
- **Preferences:** Prefers concise presentations, data-driven recommendations, and measurable business outcomes.

## Project Context
- **Initiative:** Analytics modernization initiative focused on improving reporting speed, business dashboards, customer analytics, and integration with internal systems.

## Recent Meeting Summary (August 12, 2026)
- **Dashboard:** Prototype demonstrated and officially approved by Sarah Mitchell.
- **API Integration:** Additional integration work identified, which may affect the overall timeline.
- **Pricing:** Discussed pricing for the next phase; pending a revised proposal.

## Open Action Items
- **Alex (Account Manager):** Send revised pricing proposal.
- **David (Solutions Architect):** Confirm API integra

## 31. Format the Meeting Brief

The final response is structured as a concise meeting brief so that the retrieved
information can be quickly reviewed before the client meeting.

The format highlights the client's situation, previous discussions, open actions,
key talking points, concerns, and recommended next steps.

In [39]:
def generate_meeting_brief(context):
    # Ask the LLM to organize the retrieved information
    response = llm.generate_content(
        f'''
        Prepare a concise client meeting brief using only the information provided.

        Use the following format:

        CLIENT OVERVIEW
        PREVIOUS DISCUSSIONS
        OPEN ACTION ITEMS
        KEY TALKING POINTS
        CONCERNS
        RECOMMENDED NEXT STEPS

        Retrieved information:
        {context}

        Do not invent information.
        '''
    )

    return response.text

In [40]:
def load_memories():
    # Reset existing memory data before loading
    long_term_memories.clear()

    # Create a fresh FAISS index
    global memory_index
    memory_index = faiss.IndexFlatL2(384)

    # Stop if no memory file exists
    if not MEMORY_FILE.exists():
        return

    # Read stored memories
    with open(MEMORY_FILE, 'r') as file:
        memories = [
            line.strip()
            for line in file
            if line.strip()
        ]

    # Store the memories in the current session
    long_term_memories.extend(memories)

    # Create embeddings and add them to FAISS
    if long_term_memories:
        memory_embeddings = model.encode(
            long_term_memories,
            convert_to_numpy=True
        )

        memory_index.add(memory_embeddings)

## 32. Generate the Meeting Brief

The retrieved information is passed to the language model, which organizes it into
a concise meeting brief without introducing information that was not retrieved.

In [41]:
# Retrieve information from the available sources
client_info = search_client_documents(
    'Acme Corp client profile, project status, requirements and concerns'
)

meeting_info = search_meeting_notes(
    'Acme Corp previous discussions, decisions and pending action items'
)

memory_info = search_memory(
    'Acme Corp client preferences and important information'
)

# Combine the retrieved information
retrieved_context = f'''
CLIENT INFORMATION:
{client_info}

MEETING NOTES:
{meeting_info}

LONG-TERM MEMORY:
{memory_info}
'''

# Generate the final meeting brief
meeting_brief = generate_meeting_brief(retrieved_context)

print(meeting_brief)

CLIENT OVERVIEW
- Client: Acme Corp
- Industry: E-commerce Technology (Austin, Texas; ~500 employees)
- Primary Contact: Sarah Mitchell, VP of Operations
- Account Status: Active, existing client
- Key Business Areas: E-commerce operations, customer analytics, order management, business intelligence
- Client Preferences: Prefers concise presentations, values data-driven recommendations, and is interested in measurable business outcomes

PREVIOUS DISCUSSIONS
- Discussed modernizing Acme Corp's analytics platform to address slow reporting and limited KPI visibility.
- Reviewed the initial analytics modernization proposal; Acme showed strong interest in the executive dashboard and requested a working dashboard prototype (which has since been provided and approved).
- Discussed integrating the solution with Acme's internal systems via an API (currently in progress, with additional integration work identified that may affect the timeline).
- Discussed pricing for the next phase, leading to 

## 33. Test Short-Term Memory

Short-term memory is tested through multiple messages within the same session. The
agent should retain information from earlier messages and use it when responding to
later requests.

- **Session:** A continuous interaction with the agent.

In [42]:
# Start a new conversation session
conversation_history = []

# Add the first user message
add_to_memory(
    'user',
    'I have a meeting with Acme Corp tomorrow.'
)

# Add the agent response
add_to_memory(
    'assistant',
    'I can help prepare the meeting brief.'
)

# Add a follow-up user message
add_to_memory(
    'user',
    'The meeting is focused on the analytics project.'
)

# Display the conversation history
for message in conversation_history:
    print(f"{message['role']}: {message['content']}")

user: I have a meeting with Acme Corp tomorrow.
assistant: I can help prepare the meeting brief.
user: The meeting is focused on the analytics project.


## 34. Store New Long-Term Information

Information that may be useful in future sessions can be explicitly stored as
long-term memory.

In [43]:
# Store information that should remain available in future sessions
save_memory(
    'Sarah Mitchell prefers concise presentations.'
)

save_memory(
    'Acme Corp wants measurable improvements in analytics performance.'
)

print('Stored memories:', len(long_term_memories))

Stored memories: 8


## 35. Test Long-Term Memory

The memory is retrieved using a query from a new context. This demonstrates that
stored information can be recovered independently of the current conversation.

In [44]:
# Simulate a query from a later session
memory_results = search_memory(
    'What should I know about Sarah before the Acme meeting?'
)

# Display the retrieved memories
for result in memory_results:
    print(result['memory'])

Sarah Mitchell prefers concise presentations.
Sarah Mitchell prefers concise presentations.
Sarah Mitchell prefers concise presentations.


## 36. End-to-End Meeting Preparation

The complete workflow combines retrieval, tools, short-term memory, long-term memory,
and LLM generation to prepare a meeting brief from a single user request.

The agent can retrieve information from multiple sources before producing the final
response.

In [45]:
def prepare_meeting(client):
    # Create the user's request
    user_query = f'Prepare me for my meeting with {client}.'

    # Store the request in short-term memory
    add_to_memory('user', user_query)

    # Retrieve client information
    client_info = search_client_documents(
        f'{client} profile, requirements, project status and concerns'
    )

    # Retrieve previous meeting information
    meeting_info = search_meeting_notes(
        f'{client} previous meetings, discussions and pending action items'
    )

    # Retrieve relevant long-term memories
    memory_info = search_memory(
        f'{client} client preferences and important information'
    )

    # Combine information retrieved from all sources
    context = f'''
    CLIENT INFORMATION:
    {client_info}

    MEETING NOTES:
    {meeting_info}

    LONG-TERM MEMORY:
    {memory_info}
    '''

    # Generate the final meeting brief
    meeting_brief = generate_meeting_brief(context)

    # Store the generated response in short-term memory
    add_to_memory('assistant', meeting_brief)

    return meeting_brief

In [46]:
print('Client chunks:', len([c for c in chunks if c['type'] == 'client_document']))
print('Meeting chunks:', len([c for c in chunks if c['type'] == 'meeting_note']))
print('Long-term memories:', len(long_term_memories))
print('Memory index vectors:', memory_index.ntotal)

Client chunks: 6
Meeting chunks: 6
Long-term memories: 8
Memory index vectors: 10


## 37. Run the Meeting Preparation Agent

The complete workflow is executed using the client name as input. The resulting brief
contains information retrieved from client documents, previous meeting notes, and
long-term memory.

In [47]:
# Generate a meeting brief for Acme Corp
meeting_brief = prepare_meeting('Acme Corp')

# Display the final result
print(meeting_brief)

CLIENT OVERVIEW
- Company: Acme Corp
- Industry: E-commerce Technology (Headquarters: Austin, Texas; Size: ~500 employees)
- Primary Contact: Sarah Mitchell, VP of Operations
- Focus: E-commerce operations, customer analytics, order management, and business intelligence

PREVIOUS DISCUSSIONS
- Discussed the need to modernize Acme Corp's analytics platform to address slow reporting and limited KPI visibility.
- Reviewed and demonstrated the dashboard prototype, which received positive feedback and met initial requirements.
- Discussed integrating the solution with internal systems via API, identifying additional integration work that may affect the timeline.
- Discussed pricing for the next phase.

OPEN ACTION ITEMS
- Pricing for the next phase is still under discussion.
- API integration is currently in progress.

KEY TALKING POINTS
- Improving analytics capabilities (faster reporting, better dashboards, improved internal system integration, and better visibility into customer behavior

## 38. Run the Agentic Workflow

The final workflow uses the agent to interpret the user's goal, select the required
tools, process their observations, and generate the meeting brief.

In [48]:
# Start a new short-term conversation
conversation_history = []

# Run the agent for the meeting preparation task
meeting_brief, agent_history = run_agent(
    'Prepare me for my meeting with Acme Corp.'
)

# Display the final meeting brief
print(meeting_brief)

# Meeting Brief: Acme Corp

## Client Overview
- **Company:** Acme Corp (E-commerce Technology, ~500 employees, Austin, Texas)
- **Primary Contact:** Sarah Mitchell, VP of Operations
- **Relationship:** Existing client currently undergoing an analytics modernization initiative.
- **Client Preferences:** Concise presentations, data-driven recommendations, and measurable business outcomes.

## Project Status & Recent Discussions
- **Analytics Modernization Initiative:** Focused on faster reporting, improved dashboards, customer analytics, and better internal system integration.
- **Dashboard:** The prototype was demonstrated on August 12, 2026, and officially approved by Sarah Mitchell.
- **API Integration:** Technical team identified additional integration work that may impact the timeline.
- **Pricing:** A revised pricing proposal was requested by Acme and is currently pending.

## Open Action Items from Last Meeting (August 12, 2026)
- **Alex:** Send revised pricing proposal.
- **Davi

## 39. Inspect Agent Tool Usage

The agent history is inspected to verify that the agent used its available tools
before producing the final meeting brief.

In [49]:
# Display the tool calls and observations made by the agent
for message in agent_history:

    if message['role'] in ['assistant', 'tool']:
        print(f"{message['role'].upper()}:")
        print(message['content'])
        print('-' * 50)

ASSISTANT:
TOOL: search_client_documents
QUERY: Acme Corp
--------------------------------------------------
TOOL:
[{'source': 'company_overview.txt', 'text': 'Acme Corp is an e-commerce technology company focused on helping online retailers\nmanage orders, customers, and operational workflows.\n\nThe company is currently investing in improving its analytics capabilities.\nLeadership wants better visibility into customer behavior, operational performance,\nand business KPIs.\n\nAcme Corp is evaluating improvements to its existing analytics platform and is\nparticularly interested in faster reporting, better dashboards, and improved\nintegration with internal sy', 'distance': 0.44607242941856384}, {'source': 'client_profile.txt', 'text': 'Client: Acme Corp\n\nIndustry: E-commerce Technology\n\nHeadquarters: Austin, Texas\n\nCompany Size: Approximately 500 employees\n\nPrimary Contact: Sarah Mitchell\n\nRole: VP of Operations\n\nRelationship: Existing client\n\nAccount Status: Active\n\n

## 40. Test Different Agent Tasks

Different requests are used to verify that the agent can select and use the appropriate
tools based on the information required to complete each task.

In [50]:
# Start a fresh conversation for the test
conversation_history = []

# Test a request focused on previous meetings and action items
meeting_brief, agent_history = run_agent(
    'What are the pending action items from my previous Acme Corp meetings?'
)

# Display the response
print(meeting_brief)

Based on the previous meetings with Acme Corp, the pending action items are:

- **Alex:** Send the revised pricing proposal.
- **David:** Confirm the API integration timeline.
- **Sarah:** Provide the final API documentation.


In [51]:
# Display the agent's tool calls
for message in agent_history:

    if message['role'] == 'assistant':
        print(message['content'])
        print('-' * 50)

TOOL: search_meeting_notes
QUERY: Acme Corp action items pending
--------------------------------------------------


In [52]:
# Start another fresh conversation
conversation_history = []

# Test a request focused on the client
response, agent_history = run_agent(
    'Hey can you help me recall what happened in the previous acme meetings'
)

print(response)

Here is a summary of the previous meetings with Acme Corp:

**1. Meeting on June 10, 2026**
- **Discussion:** Discussed Acme's need to modernize its analytics platform due to slow reporting and limited KPI visibility. Sarah Mitchell requested a proposal covering dashboard improvements and customer analytics.
- **Action Items:**
  - Alex: Prepare initial analytics modernization proposal.
  - Sarah: Share current reporting requirements.

**2. Meeting on July 15, 2026**
- **Discussion:** Reviewed the initial analytics modernization proposal. Acme showed strong interest in the executive dashboard and discussed integrating the solution with internal systems via an API. Sarah requested a working dashboard prototype.
- **Action Items:**
  - Alex: Provide dashboard prototype.

**3. Meeting on August 12, 2026**
- **Discussion:** Demonstrated the dashboard prototype, which received positive feedback and met initial requirements. Discussed API integration in more detail, where the technical team 